# What does compile change? Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_compile_diag_2026-11-04.md` (D1–D7). All runs share this session. About 75 min in total (~2.5 compute units). The fp32 part comes last and takes about 28 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, diffusers.__version__, transformers.__version__)

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## R: eager bf16 reference (~15 min), then D1

In [ ]:
!python bench/profile_baseline.py --frames 33 --out results/flash_33f

In [ ]:
!python bench/hash_frames.py results/flash_33f --check results/eager_33f/frames_sha256.json

## C: compile, default settings (~13 min)

A rerun, because the earlier compiled frames were not kept. It writes to a new directory, so the committed `compile_33f` stays as it is.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --kernels --out results/compile_rerun_33f

In [ ]:
!python bench/compare_frames.py --ref results/flash_33f --test results/compile_rerun_33f
!python bench/hash_frames.py results/compile_rerun_33f

## CE: compile with `emulate_precision_casts=True` (~13 min), then D2 and D3

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --emulate-precision-casts --kernels --out results/compile_emul_33f

In [ ]:
!python bench/compare_frames.py --ref results/flash_33f --test results/compile_emul_33f
!python bench/hash_frames.py results/compile_emul_33f

## F: fp32 transformer, fox and night market (~28 min), then D4–D6

bf16 weights upcast to fp32, TF32 off, memory-efficient attention. Expected peak is about 15 GB, during the VAE decode with the fp32 transformer still resident.

In [ ]:
!python bench/profile_baseline.py --frames 33 --dtype fp32 --attn efficient --prompt-ids 0 3 --kernels --out results/fp32_33f

In [ ]:
# Distance of each bf16 path to the fp32 reference (only videos 0 and 3 exist in fp32_33f)
for t in ['flash_33f', 'compile_rerun_33f', 'compile_emul_33f']:
    print('==', t)
    !python bench/compare_frames.py --ref results/fp32_33f --test results/{t} --out-name compare_vs_fp32.json
!python bench/hash_frames.py results/fp32_33f

## CLIP (D7)

Earlier runs are read from their committed mp4s; this session's runs from their .npy.

In [ ]:
!python bench/clip_score.py results/flash_33f results/efficient_33f results/seed1_33f results/compile_33f results/compile_rerun_33f results/compile_emul_33f results/fp32_33f

In [ ]:
!zip -qr diag_results.zip results -x '*.npy' && ls -lh diag_results.zip
from google.colab import files; files.download('diag_results.zip')